# Тестовое задание: графики v2

Агенты **с памятью (GRU)** уже обучены в `test_task.ipynb`, их метрики лежат в `runs/ppo_env{i}` и `runs/dsac_env{i}`.
Здесь дообучаем агентов **без памяти (MLP)** и рисуем красивые графики: линия — скользящее среднее,
полупрозрачная полоса — разброс (± стандартное отклонение в том же окне).

In [ ]:
import csv
import os
from pathlib import Path

# работаем из корня репозитория: там лежат runs/ и outputs/
if Path.cwd().name == "src_v2":
    os.chdir(Path.cwd().parent)

import matplotlib.pyplot as plt
import numpy as np
import torch

from gridrl_v2.models import DSAC, PPO

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("учимся на", DEVICE)

plt.style.use("seaborn-v0_8-darkgrid")  # серый фон с белой сеткой

SETUPS = {
    1: dict(h=5, w=5, n_colors=None, obstacle_fraction=0.0),
    2: dict(h=5, w=5, n_colors=5, obstacle_fraction=0.0),
    3: dict(h=10, w=10, n_colors=7, obstacle_fraction=0.1),
    4: dict(h=10, w=10, n_colors=4, obstacle_fraction=0.1),
}

## Обучение без памяти

Те же конфиги, что в `test_task.ipynb`, только `"memory": False`.

In [ ]:
ppo_cfg = {
    "device": DEVICE,
    "env_seed": 0,
    "seed": 0,
    "memory": False,
    "episodes": 1000,
    "episodes_per_update": 10,
    "epochs": 4,
    "episodes_per_batch": 5,
    "gamma": 0.99,
    "gae_lambda": 0.95,
    "lr": 3e-4,
    "clip": 0.2,
    "ent_coef": 0.01,
    "vf_coef": 0.5,
    "hidden": 128,
    "run_name": "ppo",
}
PPO_EPISODES = {1: 1000, 2: 1000, 3: 1500, 4: 1500}

for i in SETUPS:
    print(f"===== PPO без памяти, Env{i} =====")
    alg = PPO({**ppo_cfg, **SETUPS[i], "episodes": PPO_EPISODES[i], "run_name": f"ppo_env{i}_mlp"})
    alg.train()
    alg.save()

In [ ]:
# ВНИМАНИЕ: DSAC на картах 10x10 учится долго (на CPU — долго, может быть час и больше на одну карту)
dsac_cfg = {
    "device": DEVICE,
    "env_seed": 0,
    "seed": 0,
    "memory": False,
    "soft_q": True,
    "episodes": 500,
    "buffer_episodes": 5000,
    "learning_starts": 20,  # сколько эпизодов только копим опыт, не учась
    "updates_per_episode": 4,  # сколько обновлений после каждого эпизода
    "batch_size": 8,  # сколько случайных эпизодов из буфера в одном обновлении
    "gamma": 0.99,
    "tau": 0.005,
    "lr": 3e-4,
    "alpha": 0.005,
    "hidden": 128,
    "run_name": "dsac",
}
DSAC_EPISODES = {1: 500, 2: 500, 3: 800, 4: 800}

for i in SETUPS:
    print(f"===== DSAC без памяти, Env{i} =====")
    alg = DSAC({**dsac_cfg, **SETUPS[i], "episodes": DSAC_EPISODES[i], "run_name": f"dsac_env{i}_mlp"})
    alg.train()
    alg.save()

## Графики

`load_history` читает метрики прогона из `runs/<имя>/metrics.csv`.
`plot_curve` рисует одну кривую: среднее по последним `window` эпизодам и полосу ± стандартное отклонение.

In [ ]:
def load_history(run_name):
    # читаем metrics.csv: словарь {колонка: список чисел}; None, если прогона ещё нет
    path = Path("runs") / run_name / "metrics.csv"
    if not path.exists():
        print("нет прогона", run_name)
        return None
    with open(path) as f:
        rows = list(csv.DictReader(f))
    history = {}
    for key in rows[0]:
        values = []
        for row in rows:
            if row[key] == "True":
                values.append(1.0)
            elif row[key] == "False":
                values.append(0.0)
            else:
                values.append(float(row[key]))
        history[key] = values
    return history


def plot_curve(ax, values, label, color, window=50):
    # линия — среднее за последние window эпизодов, полоса — ± стандартное отклонение там же
    means = []
    stds = []
    for i in range(len(values)):
        last = [v for v in values[max(0, i - window + 1):i + 1] if not np.isnan(v)]
        if len(last) > 0:
            means.append(np.mean(last))
            stds.append(np.std(last))
        else:
            means.append(np.nan)
            stds.append(np.nan)
    means = np.array(means)
    stds = np.array(stds)
    episodes = np.arange(len(values))
    ax.plot(episodes, means, color=color, linewidth=2, label=label)
    ax.fill_between(episodes, means - stds, means + stds, color=color, alpha=0.3)


def save(fig, name):
    Path("outputs/plots").mkdir(parents=True, exist_ok=True)
    fig.savefig(f"outputs/plots/{name}.png", dpi=150, bbox_inches="tight")
    plt.show()


COLORS = ["tab:red", "tab:blue", "tab:green", "tab:orange"]

### Средняя отдача на всех четырёх средах (агенты с памятью)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, algo in zip(axes, ["ppo", "dsac"]):
    for i in SETUPS:
        history = load_history(f"{algo}_env{i}")
        if history is not None:
            plot_curve(ax, history["return"], f"Env{i}", COLORS[i - 1])
    ax.set_title(f"{algo.upper()} с памятью")
    ax.set_xlabel("эпизод")
    ax.set_ylabel("средняя отдача")
    ax.legend()
save(fig, "v2_return_all_envs")

### С памятью и без памяти

In [ ]:
for algo in ["ppo", "dsac"]:
    fig, axes = plt.subplots(1, 4, figsize=(22, 4.5))
    for i in SETUPS:
        ax = axes[i - 1]
        gru = load_history(f"{algo}_env{i}")
        mlp = load_history(f"{algo}_env{i}_mlp")
        if gru is not None:
            plot_curve(ax, gru["return"], "с памятью (GRU)", "tab:red")
        if mlp is not None:
            plot_curve(ax, mlp["return"], "без памяти (MLP)", "tab:blue")
        ax.set_title(f"{algo.upper()}, Env{i}")
        ax.set_xlabel("эпизод")
        ax.set_ylabel("средняя отдача")
        ax.legend()
    save(fig, f"v2_{algo}_memory")

### Остальные графики из задания: лосс, его компоненты, энтропия, вероятность исследования

In [ ]:
METRICS = {
    "ppo": {"loss": "полный лосс", "policy_loss": "лосс актора", "value_loss": "лосс критика",
            "entropy": "энтропия", "explore_prob": "вероятность нежадного действия", "length": "длина эпизода"},
    "dsac": {"loss": "полный лосс", "actor_loss": "лосс актора", "q_loss": "лосс критиков",
             "entropy": "энтропия", "explore_prob": "вероятность нежадного действия", "length": "длина эпизода"},
}

for algo in ["ppo", "dsac"]:
    fig, axes = plt.subplots(2, 3, figsize=(20, 10))
    axes = axes.ravel()
    for ax, (key, title) in zip(axes, METRICS[algo].items()):
        for i in SETUPS:
            history = load_history(f"{algo}_env{i}")
            if history is not None:
                plot_curve(ax, history[key], f"Env{i}", COLORS[i - 1])
        ax.set_title(f"{algo.upper()}: {title}")
        ax.set_xlabel("эпизод")
        ax.legend()
        if key == "length":
            ax.set_yscale("log")
    fig.tight_layout()
    save(fig, f"v2_{algo}_metrics")